 # Taller polars y parquet avanzado

**John Gonzalez**

*19 de octubre de 2026*

---

## Objetivo

Imaginen que ustedes trabajan en una empresa que procesan datos bigdata, un ing junior genera código con IA, pero está lleno de problemas. Tu objetivo es corregirlos.

## Entorno de Trabajo

* **Dataset:** Predicción de tarifas de Taxi de Nueva York (5.4 GB en formato CSV, descárgala de kaggle siguiendo las instrucciones vistas en clase, use como referencia el notebook 05).

* **Entorno:** GitHub Codespaces (Límite de recursos: 8 GB RAM).

## A tener en cuenta

* Cuando se le pida "consulte ..." puede apoyarse de la IA para investigar y entender, muy recomendable.
* Cuando se le pida "escriba código" puede apoyarse de la IA, pero debe entender 100% el código generado, se validará en clase el entendimiento, posiblmente se solicite el prompt utilizado para llegar a la solución.
* Cuando se le pida "escriba con sus propias palabras ..." puede parafrasear pero lo escrito debe ser 100% escrito por ud mismo.
* Este taller es la continuación del 07, se recomienda ejecutarlo y entenderlo antes de iniciar.
* Este taller aborda nuevos conceptos, es intencional.

In [19]:
# Configuración del Entorno e Importación de Librerías
import polars as pl
import os
import time

# Rutas de archivos de trabajo
path = "/home/vscode/.cache/kagglehub/competitions/new-york-city-taxi-fare-prediction"
archivo_csv = path + "/train.csv"
archivo_parquet = path + "/train.parquet"

print("Versión de Polars instalada:", pl.__version__)


Versión de Polars instalada: 2.0.0


 ---

 ## Caso 1: Evaluación Ansiosa vs. Perezosa (Gestión de Memoria RAM)



 **Escenario:** Un analista intenta cargar el archivo completo `train.csv` (5.4 GB) a la memoria RAM utilizando evaluación ansiosa (*Eager Execution*) para calcular la tarifa promedio (`fare_amount`). El sistema colapsa debido a un desbordamiento de memoria (*Out of Memory - OOM*).

 > **Tip:** En clase vimos el método `scan_csv`. Puedes ejecutar el cálculo por lotes en disco (*Out-of-Core*) agregando el argumento `streaming=True` al método `.collect()`.

In [ ]:
# CÓDIGO INEFICIENTE (CASO 1)
print("--- Caso 1: Intentando lectura ansiosa (Eager) ---")
try:
    inicio = time.time()
    # Ineficiencia: Carga ansiosa que sobrepasa la memoria física disponible
    df = pl.read_csv(archivo_csv)
    media_tarifa = df.select(pl.col("fare_amount").mean())
    print(f"Resultado: {media_tarifa} | Tiempo de ejecución: {time.time() - inicio:.2f} s")
except Exception as e:
    print(f"Error de ejecución detectado (OOM): {e}")


 ### [Espacio de Respuesta - Caso 1]

 **Escribe tu código corregido a continuación y documenta:**

 1. El tiempo de ejecución sin streaming con `.collect()`.
 2. El tiempo de ejecución alcanzado con la solución perezosa en modo *streaming* con `.collect(streaming=True)`.
 3. Consulta y explica con tus propias palabras, por qué el procesamiento por lotes (*Out-of-Core*) podría evitar el colapso de la memoria RAM.

In [20]:
# Escribe tu código aquí para el caso 1
inicio = time.time()

media_tarifa_lazy = (
    pl.scan_csv(archivo_csv)
    .select(pl.col("fare_amount").mean())
    .collect()
)

tiempo_lazy = time.time() - inicio
print(f"Resultado sin streaming: {media_tarifa_lazy.item(0,0):.2f} | Tiempo: {tiempo_lazy:.2f} s")

Resultado sin streaming: 11.35 | Tiempo: 30.35 s


In [ ]:
# Escribe tu código aquí para el caso 2
inicio = time.time()

media_tarifa_streaming = (
    pl.scan_csv(archivo_csv)
    .select(pl.col("fare_amount").mean())
    .collect(engine="streaming")
)

tiempo_streaming = time.time() - inicio
print(f"Resultado con streaming: {media_tarifa_streaming.item(0,0):.2f} | Tiempo: {tiempo_streaming:.2f} s")

Resultado con streaming: 11.35 | Tiempo: 30.25 s


## tengo que modificar esto

El procesamiento por lotes (Out-of-Core o streaming) evita que la memoria RAM colapse porque, en lugar de intentar subir los 5.4 GB del archivo a la memoria al mismo tiempo, divide los datos en fragmentos pequeños (lotes o chunks). Polars lee un lote del disco, calcula la suma y el conteo parcial de fare_amount, guarda ese pequeño resultado parcial y libera la memoria de ese lote antes de pasar al siguiente. Así, el uso de RAM se mantiene siempre bajo y constante.

 ---

 ## Caso 2: Inferencia de Tipos y Filtros Temporales (Fechas como String vs. Datetime)



 **Escenario:** Para filtrar las transacciones correspondientes al mes de diciembre de 2015, se aplican expresiones regulares sobre la columna `pickup_datetime` leída como texto (`String`). La consulta requiere escanear millones de cadenas de caracteres, resultando extremadamente lenta.

 > **Tip:** No alcazamos a ver como convertir texto a formato fecha-hora. Pero es clave porque filtrar cadenas de texto en conjuntos de datos masivos inhabilita las optimizaciones vectoriales. Convierte la columna a formato temporal nativo mediante `.str.to_datetime("%Y-%m-%d %H:%M:%S UTC")` o `.cast(pl.Datetime)`. Una vez casteada, utiliza expresiones temporales nativas como `.dt.year() == 2015` y `.dt.month() == 12`.

In [ ]:
# CODIGO INEFICIENTE (CASO 2)
print("--- Caso 2: Búsqueda sobre cadenas de texto ---")
inicio = time.time()

# Ineficiencia: Coincidencia de cadenas de texto sobre 55 millones de registros
resultado_cadenas = (
    pl.scan_csv(archivo_csv)
    .filter(pl.col("pickup_datetime").str.contains("2015-12"))
    .select(pl.len())
    .collect(streaming=True)
)

tiempo_cadenas = time.time() - inicio
print(f"Registros encontrados en Dic 2015: {resultado_cadenas.item(0,0)} | Tiempo: {tiempo_cadenas:.2f} s")


 ### [Espacio de Respuesta - Caso 2]

 **Escribe tu código corregido a continuación y documenta:**

 1. El tiempo de ejecución tras convertir la columna a `Datetime`.

 2. La ganancia porcentual en velocidad frente al escaneo de cadenas de texto.

In [26]:
#checando
print("--- Caso 2: Búsqueda sobre cadenas de texto ---")
inicio = time.time()

resultado_cadenas = (
    pl.scan_csv(archivo_csv)
    .filter(pl.col("pickup_datetime").str.contains("2015-12"))
    .select(pl.len())
    .collect(engine="streaming")
)

tiempo_cadenas = time.time() - inicio
print(f"Registros encontrados en Dic 2015: {resultado_cadenas.item(0,0)} | Tiempo: {tiempo_cadenas:.2f} s")

--- Caso 2: Búsqueda sobre cadenas de texto ---
Registros encontrados en Dic 2015: 0 | Tiempo: 30.65 s


In [27]:
# Escribe tu código aquí
inicio = time.time()

resultado_fechas = (
    pl.scan_csv(archivo_csv)
    .with_columns(
        pl.col("pickup_datetime").str.to_datetime("%Y-%m-%d %H:%M:%S UTC")
    )
    .filter(
        (pl.col("pickup_datetime").dt.year() == 2015) & 
        (pl.col("pickup_datetime").dt.month() == 12)
    )
    .select(pl.len())
    .collect(engine="streaming")
)

tiempo_fechas = time.time() - inicio
print(f"Registros encontrados en Dic 2015: {resultado_fechas.item(0,0)} | Tiempo: {tiempo_fechas:.2f} s")



Registros encontrados en Dic 2015: 0 | Tiempo: 32.02 s


 ---

 ## Caso 3: Preservación del Plan Lógico de Ejecución (DAG y `.explain()`)



 **Escenario:** En la línea inicial del pipeline, se convierte el `LazyFrame` a `DataFrame` ejecutando `.collect()`. Posteriormente, se aplican los filtros geográficos y la selección de columnas sobre el objeto en memoria.



 > **Recomendación:** Similar a `Spark`, materializar los datos usando `collect` de forma prematura destruye el Grafo Acíclico Dirigido (DAG). Esto impide que el optimizador de Polars aplique *Projection Pushdown* (seleccionar únicamente las columnas requeridas desde el almacenamiento) y *Selection Pushdown* (filtrar filas antes de cargar los datos). Conserva la cadena como `LazyFrame` e inspecciona el plan optimizado invocando `.explain()`.

In [ ]:
# CÓDIGO INEFICIENTE (CASO 3)
print("--- Caso 3: Materialización prematura del DataFrame ---")
inicio = time.time()

# Ineficiencia: Invocación prematura de .collect() que anula las optimizaciones del DAG
# NOTA: Es muy posible que esta linea desborde la memoria y tengas que reiniciar el kernel
df_materializado = pl.scan_parquet(archivo_parquet).collect()

# Las operaciones de filtrado y proyección se ejecutan de forma ansiosa sobre todo el conjunto
resultado_suboptimo = (
    df_materializado
    .filter(pl.col("passenger_count") == 1)
    .select("fare_amount")
)

tiempo_suboptimo = time.time() - inicio
print(f"Tiempo total con materialización prematura: {tiempo_suboptimo:.2f} s")


 ### [Espacio de Respuesta - Caso 3]

 **Escribe tu código corregido a continuación y documenta:**

 1. El código reordenado donde `.collect()` se ubica únicamente al final del procesamiento.

 2. La salida del comando `print(lazy_query.explain())`, señalando dónde el motor aplicó *Selection Pushdown* y *Projection Pushdown*.

In [ ]:
# Escribe el código del punto 1 aqui

In [ ]:
# Escribe el código del punto 2 aqui

 ---

 ## Caso 4: Particionamiento Físico y el "Small File Problem"



 **Escenario:** Con el propósito de organizar un Data Lake en disco, se intenta particionar el conjunto de datos en formato Parquet utilizando la columna de coordenada de longitud (`pickup_longitude`).

En clase vimos que los archivos parquet se dividen en bloques. Esto es muy beneficioso porque cada bloque tiene metadatos como el valor míni y máximo, si por ejemplo tenemos un filtro como `ickup_longitude < -74`, polars leerá solo los bloques que cumplen con esta condición. Además, también existe la posibilidad de dividir los datos en pequeños archivos o carpetas parquet, de tal manera que polars solo se enfocará en leer aquellos archivos que cumplen alguna condición por ejemplo `year=2015`. 


 > **Recomendación:** Polars ofrece la oportunidad de particionar por variable, sin embargo, cuando la partición se hace por variables continuas puede generar miles de directorios y archivos Parquet diminutos (de pocos kilobytes). Esto provoca latencias severas en el sistema de archivos (Disk I/O Latency). Selecciona columnas discretas o categóricas de **baja cardinalidad** (ej. `year` y `month`) para mantener tamaños óptimos (100 MB - 500 MB).

In [ ]:
# CÓDIGO INEFICIENTE (CASO 4)
print("--- Caso 4: Particionamiento por variable continua (Alta Cardinalidad) ---")
# ADVERTENCIA: La siguiente llamada se limita conceptualmente para evitar la creación masiva de archivos.
try:
    # Ineficiencia: Intento de particionamiento sobre una columna flotante continua
    pl.scan_parquet(archivo_parquet).sink_parquet(
        "parquet_inadecuado",
        partition_by=["pickup_longitude"]
    )
except Exception as e:
    print(f"Respuesta del sistema ante particionamiento inadecuado: {e}")


 ### [Espacio de Respuesta - Caso 4]

 **Escribe tu código corregido a continuación y documenta:**

 1. El código que particiona por variables discretas adecuadas (por ejemplo, extraídas de las fechas).

 2. Un comando de terminal o script que verifique la estructura de directorios y el tamaño de los archivos generados. Recuerda que en jupyter para ejecutar un comando de consola se usa el símbolo `!` al inicio de la celda.

In [4]:
# Escribe tu código aquí para el punto 1 aquí

In [5]:
# Escribe los comandos de terminal aquí

 ---

 ## Caso 5: Resiliencia de Esquemas y Valores Faltantes en Producción



 **Escenario:** Al procesar un nuevo archivo de datos, la inferencia automática de esquemas falla o produce comportamientos inconsistentes debido a la presencia de valores nulos e incoherencias de tipos en la columna `passenger_count`.



 > **Recomendación:** En entornos de producción no se debe confiar ciegamente en la inferencia automática de esquemas sobre archivos planos (CSV). Utiliza la opción `schema_overrides` dentro de `scan_csv()` para definir explícitamente el esquema de datos y aplica expresiones de imputación (`.fill_null()`) o filtrado (`.drop_nulls()`) dentro de la consulta perezosa.

In [ ]:
# CÓDIGO INEFICIENTE (CASO 5)
print("--- Caso 5: Inferencia rígida de esquema sin tratamiento de nulos ---")
try:
    # Ineficiencia: Dependencia de la inferencia de tipos sin manejo de esquema ni nulos
    resultado_esquema = (
        pl.scan_csv(archivo_csv)
        .select(pl.col("passenger_count").mean())
        .collect(streaming=True)
    )
    print("Promedio de pasajeros:", resultado_esquema)
except Exception as e:
    print(f"Error detectado en la inferencia de esquema: {e}")


 ### [Espacio de Respuesta - Caso 5]

 **Escribe tu código corregido a continuación y documenta:**

 1. Consulta como usar `schema_overrides`.
 2. Define explícitamente los tipos usando `schema_overrides`.
 3. Aplica la estrategia para la gestión de valores nulos dentro del marco perezoso.

Escribe con tus propias palabras que es y como usar `schema_overrides`

In [ ]:
# Escribe tu código aquí para el punto 2

In [ ]:
# Escribe tu código aquí para el punto 3

 ---

 ## Caso 6: Interoperabilidad Eficiente en Machine Learning (Apache Arrow / Zero-Copy)

 **Escenario:** Al preparar la matriz de características para entrenar un modelo predictivo en la librería ara machine learning `Scikit-Learn`, el analista invoca la función `.to_pandas()`, provocando una duplicación innecesaria del consumo de memoria RAM.

 > **Recomendación:** Polars gestiona sus datos internamente mediante el formato de memoria **Apache Arrow**. La conversión a Pandas genera una copia completa en memoria bajo bloques de NumPy. Para pasar datos a algoritmos de Machine Learning de forma eficiente, utiliza la exportación nativa a tablas Arrow (`.to_arrow()`) o arreglos NumPy directos, aprovechando la transferencia sin copia (*Zero-Copy*).

In [ ]:
# CÓDIGO INEFICIENTE (CASO 6)
print("--- Caso 6: Duplicación de memoria al convertir a Pandas ---")
inicio = time.time()

# Ineficiencia: Carga a Polars y posterior conversión ansiosa a Pandas
df_polars = pl.scan_parquet(archivo_parquet).select(["fare_amount", "passenger_count"]).collect()
df_pandas = df_polars.to_pandas()  # Genera duplicación de datos en RAM

tiempo_conversion = time.time() - inicio
print(f"Matriz procesada y convertida a Pandas en {tiempo_conversion:.2f} s")


 ### [Espacio de Respuesta - Caso 6]

 **Escribe tu código corregido a continuación y documenta:**

 1. Escribe el código para la exportación a Apache Arrow o arreglos NumPy directa.
 2. Explica conceptualmente con tus propias palabras cómo la especificación de memoria de Apache Arrow evita la duplicación de bytes en la RAM.

In [3]:
# Escribe tu código aquí

Explica el punto 2 aquí con tus propias palabras

---

Felicitaciones, haz terminado el taller, ahora entiendes los conceptos claves de cualquier ingeniero de datos que trabaja en una empresa mediana-grande.